<a href="https://www.kaggle.com/code/avikdas567/rental-product-recommendation-system?scriptVersionId=295899637" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import gc
import warnings
import re

# Suppress warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURATION
# ==========================================
BASE_PATH = '/kaggle/input/rental-product-recommendation-system/'
FILES = {
    'hits': BASE_PATH + 'metrika_hits_test.csv',
    'visits': BASE_PATH + 'metrika_visits_test.csv',
    'products': BASE_PATH + 'new_site_products.csv',
    'new_orders': BASE_PATH + 'new_site_orders.csv',
    'old_orders': BASE_PATH + 'old_site_orders.csv',
    'old_carts': BASE_PATH + 'old_site_carts.csv',
    'mapping': BASE_PATH + 'old_site_new_site_products.csv'
}

# Weights for the Ensemble (Heuristic Tuning)
W_ORDER = 5.0   # Strongest signal: Bought together
W_CART = 3.0    # Strong signal: Added to cart together
W_VIEW = 1.0    # Moderate signal: Viewed together

print("Initializing High-Rank Solution with Cross-Site Learning...")

# ==========================================
# 1. LOAD MAPPINGS
# ==========================================
print("Loading ID Mappings...")

# A. Slug -> ID (New Site)
products_df = pd.read_csv(FILES['products'], usecols=['id', 'slug'])
slug_to_id = dict(zip(products_df['slug'], products_df['id']))
# Keep global popularity fallback
all_product_ids = products_df['id'].unique()
del products_df

# B. Old ID -> New ID (Cross-Site)
mapping_df = pd.read_csv(FILES['mapping'])
old_to_new_id = dict(zip(mapping_df['old_site_id'], mapping_df['new_site_id']))
del mapping_df

gc.collect()

# ==========================================
# 2. HELPER TO UPDATE MATRICES
# ==========================================
# We use a nested dict structure: matrix[item_a][item_b] = score
co_matrix = defaultdict(lambda: defaultdict(float))

def update_matrix(groups, weight_multiplier):
    """Updates the global co-occurrence matrix."""
    for items in groups:
        if len(items) > 1:
            # Normalize by group size (pairs in huge orders are less significant)
            w = weight_multiplier / (len(items) - 1)
            for i in items:
                for j in items:
                    if i != j:
                        co_matrix[i][j] += w

# ==========================================
# 3. PROCESS ORDERS (New + Old)
# ==========================================
print("Processing Orders (New & Old)...")

# A. New Site Orders
new_orders = pd.read_csv(FILES['new_orders'], usecols=['product_id', 'number'])
# Global Best Sellers (Backup)
top_products = new_orders['product_id'].value_counts().head(6).index.tolist()
# Update Matrix
new_groups = new_orders.groupby('number')['product_id'].apply(list)
update_matrix(new_groups, W_ORDER)
del new_orders, new_groups
gc.collect()

# B. Old Site Orders (Mapped)
old_orders = pd.read_csv(FILES['old_orders'], usecols=['product_id', 'user_id', 'create_date']) 

# Create a "basket_id" to group items bought by the same user on the same day
old_orders['basket_id'] = old_orders['user_id'].astype(str) + '_' + old_orders['create_date']
old_orders['new_product_id'] = old_orders['product_id'].map(old_to_new_id)
old_orders = old_orders.dropna(subset=['new_product_id'])

old_groups = old_orders.groupby('basket_id')['new_product_id'].apply(list)
update_matrix(old_groups, W_ORDER)
del old_orders, old_groups
gc.collect()

# ==========================================
# 4. PROCESS CARTS (Old Site)
# ==========================================
print("Processing Old Site Carts...")
old_carts = pd.read_csv(FILES['old_carts'], usecols=['user_id', 'product_id'])
# Group by user (approximate cart session)
old_carts['new_product_id'] = old_carts['product_id'].map(old_to_new_id)
old_carts = old_carts.dropna(subset=['new_product_id'])

cart_groups = old_carts.groupby('user_id')['new_product_id'].apply(list)
update_matrix(cart_groups, W_CART)
del old_carts, cart_groups
gc.collect()

# ==========================================
# 5. PROCESS NEW SITE VIEWS (Co-visitation)
# ==========================================
print("Processing Current Session Views...")

# A. Map Hits
hits_df = pd.read_csv(FILES['hits'], usecols=['watch_id', 'slug'], dtype={'watch_id': str})
hits_df['product_id'] = hits_df['slug'].map(slug_to_id)
hits_df = hits_df.dropna(subset=['product_id'])
watch_id_to_product = dict(zip(hits_df['watch_id'], hits_df['product_id'].astype(int)))
del hits_df
gc.collect()

# B. Process Sessions
visits_df = pd.read_csv(FILES['visits'], usecols=['visit_id', 'watch_ids'])
pattern = re.compile(r'\d+') 
session_histories = {} 

print("Building View Matrix...")
for row in visits_df.itertuples(index=False):
    visit_id = row[0]
    raw_ids = pattern.findall(str(row[1]))
    
    history = []
    for wid in raw_ids:
        if wid in watch_id_to_product:
            history.append(watch_id_to_product[wid])
    
    session_histories[visit_id] = history
    
    # Update Matrix with Time-Decay
    if len(history) > 1:
        for i in range(len(history)):
            item_a = history[i]
            # Window +/- 2
            start = max(0, i-2)
            end = min(len(history), i+3)
            for j in range(start, end):
                if i != j:
                    item_b = history[j]
                    diff = abs(i - j)
                    # Decay: Neighbor=1.0, Gap of 1=0.5
                    w = W_VIEW / diff 
                    co_matrix[item_a][item_b] += w

# ==========================================
# 6. GENERATE PREDICTIONS
# ==========================================
print("Generating Final Predictions...")
final_preds = []

# Pre-calculate global best string for speed
global_fill = top_products[:6]

for visit_id in visits_df['visit_id']:
    history = session_histories.get(visit_id, [])
    scores = defaultdict(float)
    seen_items = set(history)
    
    if history:
        # We weight the last 3 items, giving huge priority to the very last one
        recent_items = history[-3:]
        
        for i, item in enumerate(reversed(recent_items)):
            # i=0 is last item. Weight decay: 1.0, 0.7, 0.4
            pos_weight = 1.0 - (i * 0.3)
            if pos_weight <= 0: break
            
            if item in co_matrix:
                # Retrieve neighbors from our massive unified matrix
                neighbors = co_matrix[item]
                for cand, score in neighbors.items():
                    if cand not in seen_items:
                        scores[cand] += score * pos_weight

    # Select Top 6
    if scores:
        best_candidates = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        prediction = [x[0] for x in best_candidates[:6]]
    else:
        prediction = []
    
    # Fill with Global Popularity if short
    if len(prediction) < 6:
        for item in global_fill:
            if item not in prediction and item not in seen_items:
                prediction.append(item)
            if len(prediction) == 6:
                break
                
    # Final check to ensure exactly 6 (rare edge case where seen_items excludes everything)
    if len(prediction) < 6:
        for item in global_fill:
            if item not in prediction:
                prediction.append(item)
            if len(prediction) == 6:
                break

    final_preds.append(" ".join(map(str, prediction)))

# ==========================================
# 7. SUBMISSION
# ==========================================
visits_df['product_ids'] = final_preds
submission = visits_df[['visit_id', 'product_ids']]
submission.to_csv('submission.csv', index=False)

print("Success! 'submission.csv' created.")
display(submission.head())

Initializing High-Rank Solution with Cross-Site Learning...
Loading ID Mappings...
Processing Orders (New & Old)...
Processing Old Site Carts...
Processing Current Session Views...
Building View Matrix...
Generating Final Predictions...
Success! 'submission.csv' created.


,visit_id,product_ids
0,6034151852304141635,463480429 463480693 495400618 463480694 495401...
1,0473312698303184850,495403354.0 495520959.0 495507034.0 463480381....
2,0140437049255950634,463480255 463480256 463480236 463480466 463480...
3,4595976301611479438,463480429 463480693 495400618 463480694 495401...
4,1835293676913553579,495277589 463480429 463480225 463480224 463480...
